In [3]:
import os

os.getcwd()

'/media/hamza/Disque local1/Hamza Bouajila/Portfolio/Building-a-Batch-ETL-Pipeline-RedPanda/notebooks'

In [4]:
os.chdir("../")
os.getcwd()

'/media/hamza/Disque local1/Hamza Bouajila/Portfolio/Building-a-Batch-ETL-Pipeline-RedPanda'

In [9]:
import quixstreams as qx
from quixstreams.models import TopicAdmin

from data_pipeline.src.core.config import settings
from data_pipeline.src.core.logging_config import setup_logging


class RedpandaBase:
    def __init__(self):
        self.logger = setup_logging("RedpandaBase")
        self.app = qx.Application(
            broker_address="localhost:9092",
            consumer_group="StreamingAppConsumerGroup",
            auto_offset_reset="earliest",
            loglevel=settings.LOG_LEVEL,
            request_timeout=5,
        )
        self.topic_admin = TopicAdmin(settings.REDPANDA_BROKER_ADDRESS)
        self.bitcoin_input_topic = self.app.topic(
            settings.REDPANDA_BITCOIN_INPUT_TOPIC, value_deserializer="json"
        )

        self.bitcoin_output_topic = self.app.topic(
            settings.REDPANDA_BITCOIN_OUTPUT_TOPIC, value_serializer="json"
        )

        self.news_input_topic = self.app.topic(
            settings.REDPANDA_NEWS_INPUT_TOPIC, value_deserializer="json"
        )
        self.news_output_topic = self.app.topic(
            settings.REDPANDA_NEWS_OUTPUT_TOPIC, value_serializer="json"
        )

    def clear_topics(self):
        # Delete the topics
        self.topic_admin.admin_client.delete_topics(
            [
                settings.REDPANDA_NEWS_INPUT_TOPIC,
                settings.REDPANDA_NEWS_OUTPUT_TOPIC,
                settings.REDPANDA_BITCOIN_INPUT_TOPIC,
                settings.REDPANDA_BITCOIN_OUTPUT_TOPIC,
            ]
        )

    def run(self):
        self.app.run()

In [10]:
class RedpandaConsumer(RedpandaBase):
    def __init__(self):
        super().__init__()
        self.logger.name = "RedpandaConsumer"

    def bitcoin_consume_data(self):
        messages = []
        with self.app.get_consumer() as consumer:
            consumer.subscribe([self.bitcoin_output_topic.name])
            self.logger.info(
                f"Consumer Subscribed to topic: {self.bitcoin_output_topic.name}"
            )

            while True:
                message = consumer.poll(timeout=10)
                if message is None:
                    self.logger.info("No more messages received.")
                    break
                elif message.error() is not None:
                    self.logger.error(f"Error: {message.error()}")
                    raise Exception(f"Error: {message.error()}")
                else:
                    messages.append({"Key": message.key(), "Value": message.value()})
                    self.logger.info(f"Message received: {message.value()}")

        return messages

    def news_consume_data(self):
        messages = []
        with self.app.get_consumer() as consumer:
            consumer.subscribe([self.news_output_topic.name])
            self.logger.info(
                f"Consumer Subscribed to topic: {self.news_output_topic.name}"
            )

            while True:
                message = consumer.poll(timeout=10)
                if message is None:
                    self.logger.info("No more messages received.")
                    break
                elif message.error() is not None:
                    self.logger.error(f"Error: {message.error()}")
                    raise Exception(f"Error: {message.error()}")
                else:
                    messages.append({"Key": message.key(), "Value": message.value()})
                    self.logger.info(f"Message received: {message.value()}")

        return messages

In [11]:
consumer = RedpandaConsumer()
message = consumer.bitcoin_consume_data()

[2025-01-13 22:52:13,654] [INFO] [quixstreams] : Topics required for this application: "BitcoinRawData", "BitcoinCleanedData", "NewsRawData", "NewsCleanedData"
[2025-01-13 22:52:13,658] [INFO] [quixstreams] : Validating Kafka topics exist and are configured correctly...
[2025-01-13 22:52:13,668] [INFO] [quixstreams] : Kafka topics validation complete
[2025-01-13 22:52:13,693] [DEBUG] [quixstreams] : Assigning topic partition "BitcoinCleanedData[0]"


[2025-01-13 22:52:23,887] [DEBUG] [quixstreams] : Closing Kafka consumer
[2025-01-13 22:52:23,889] [DEBUG] [quixstreams] : Revoking topic partition "BitcoinCleanedData[0]"
[2025-01-13 22:52:23,891] [DEBUG] [quixstreams] : Kafka consumer closed


In [15]:
message[0]

{'Key': b'939c0935-09be-41e9-84be-85eec1021207',
 'Value': b'[{"price":0.9683475437700433,"volume_24h":5266,"volume_change_24h":0.4737719336411441,"percent_change_1h":0.3328455289313872,"percent_change_24h":0.48129848730279257,"percent_change_7d":0.18025600917173445,"market_cap":0.4203409247720509,"market_cap_dominance":1975,"fully_diluted_market_cap":0.08547953372708506,"last_updated":"2025-01-13T20:52:39.351Z","id":"1719c979425945a88110917348556d72"},{"price":0.24435896320381834,"volume_24h":7468,"volume_change_24h":0.09512010470449361,"percent_change_1h":0.8929802495743291,"percent_change_24h":0.38381929626136047,"percent_change_7d":0.5672167651045816,"market_cap":0.03506791384763708,"market_cap_dominance":2359,"fully_diluted_market_cap":0.5929919049777206,"last_updated":"2025-01-13T20:52:39.351Z","id":"42b1bdc55b794a1c88d95198df16981d"},{"price":0.8746847131082713,"volume_24h":3621,"volume_change_24h":0.042605464197286835,"percent_change_1h":0.33110344527419344,"percent_change_24h"